# lineplot

手册 5.5.1 菜单 1 的九图: 包络/散角/发射度/束长/能散/能量/纵向发射度 + 参考粒子轨迹。

In [ ]:
%run _bootstrap.py

In [ ]:
from astra_tools.widgets.selectors import discover_sim_runs
from astra_tools.io.astra_emit import read_emit_files, read_ref_file
runs = discover_sim_runs(SIM_DIR)
if not runs:
    from IPython.display import display, HTML
    display(HTML("<div style='background:#ffe0e0;border:1px solid #e57373;padding:8px 12px'>未找到 ASTRA 输出文件 — 请先运行 02_astra.ipynb 完成一次追踪</div>"))
    raise SystemExit("未找到 ASTRA 输出 (请先运行 02_astra)")
stem = sorted(runs)[0]
# R2-2-4: max(..., key=int) 取代 sorted()[-1] — run>9 时字典序错误
# (discover_sim_runs 只收集纯数字 run; 非数字键按 0 容错)
run_key = max(runs[stem], key=lambda r: int(r) if r.isdigit() else 0)
print("使用 stem:", stem, "| run:", run_key)
emit = read_emit_files(str(SIM_DIR / stem), run=run_key)
ref = read_ref_file(str(SIM_DIR / stem), run=run_key)
print("Xemit/Yemit/Zemit 行数:", len(emit.x.z), len(emit.y.z), len(emit.z.z))

In [ ]:
from astra_tools.plot.emit_plots import plot_lineplot_overview
plot_lineplot_overview(emit)

In [ ]:
from astra_tools.plot.emit_plots import plot_emittance_evolution, plot_energy_evolution
plot_emittance_evolution(emit)
plot_energy_evolution(emit)

In [ ]:
from astra_tools.plot.emit_plots import plot_ref_trajectory
plot_ref_trajectory(ref)

In [ ]:
# 光学函数: beta/alpha、相位推进、相干长度 (菜单 2)
from astra_tools.plot.advanced_plots import (
    plot_beta_alpha, plot_phase_advance, plot_coherence_length,
)
plot_beta_alpha(emit, ref=ref)
plot_phase_advance(emit, ref=ref)
plot_coherence_length(emit)

In [ ]:
# 核心发射度 (Cemit) 与缩减发射度 (Xemit2), 存在则显示 (菜单 4)
from astra_tools.io import parse_output_file, read_xemit2
from astra_tools.plot.advanced_plots import plot_core_emittance, plot_reduced_emittance, plot_trace_emittance
from pathlib import Path

# R2-2-4: 全部用 cell 2 动态选择的 stem/run_key, 不再硬编码 .001
try:
    ce = parse_output_file(str(SIM_DIR / (stem + ".Cemit." + run_key)))
    plot_core_emittance(ce)
except Exception:
    print("无 Cemit 文件 (OUTPUT 中 C_EmitS=F)")
try:
    x2 = read_xemit2(str(SIM_DIR / (stem + ".Xemit2." + run_key)))
    y2 = read_xemit2(str(SIM_DIR / (stem + ".Yemit2." + run_key)))
    plot_reduced_emittance(x2, y2)
except Exception:
    print("无 Xemit2 文件 (OUTPUT 中 Lsub_cor=F)")

In [ ]:
# 探针轨迹与空间电荷场 (存在 track 文件时)
from astra_tools.plot.advanced_plots import plot_probe_trajectories, plot_space_charge_fields
from pathlib import Path
tr = SIM_DIR / (stem + ".track." + run_key)   # R2-2-4: 动态 stem/run_key
if tr.exists():
    plot_probe_trajectories(tr)
    plot_space_charge_fields(tr, "Ez")
else:
    print("无 track 文件 (OUTPUT 中 TrackS=F)")

In [ ]:
from astra_tools.io.astra_emit import read_sigma_file
from astra_tools.plot.emit_plots import plot_eigen_emittances
try:
    # R2-2-4: 动态 stem + run_key (read_sigma_file 按 run 拼接文件名)
    plot_eigen_emittances(read_sigma_file(str(SIM_DIR / stem), run=run_key))
except FileNotFoundError:
    print("无 Sigma 文件 (OUTPUT 中 SigmaS=F)")

## 粒子速度与平均步长 (lineplot 菜单 2)

In [ ]:
# 参考粒子速度 beta=v/c 与 gamma (由 pz 推出)
from astra_tools.plot.emit_plots import plot_velocity_evolution, plot_step_size_evolution
plot_velocity_evolution(ref)

In [ ]:
# 平均积分步长 (ref 文件相邻行的 z 间距)
plot_step_size_evolution(ref)

## 时间轴变体 (三视图 vs 时间): 把 x_axis 换成 't'

In [ ]:
from astra_tools.plot.emit_plots import (
    plot_envelope_evolution, plot_emittance_evolution,
    plot_bunch_length_evolution, plot_energy_spread_evolution)
plot_envelope_evolution(emit, x_axis="t")

In [ ]:
plot_emittance_evolution(emit, x_axis='t')

In [ ]:
plot_bunch_length_evolution(emit, x_axis='t')

### 菜单 1/2 补充: 关联能散、参考动量、压缩(时间)

In [ ]:
from astra_tools.plot.emit_plots import (
    plot_correlated_energy_spread, plot_ref_momentum)
plot_correlated_energy_spread(emit)
plot_ref_momentum(ref)

from pathlib import Path as _P
from astra_tools.io.astra_misc import read_pscan, read_scan, read_tcheck
from astra_tools.plot.advanced_plots import (
    plot_pscan_compression_time, plot_scan_fom, plot_scan_position,
    plot_tcheck_counter)
# R2-2-4: 全部用 cell 2 动态选择的 stem/run_key, 不再硬编码 .001
ps = SIM_DIR / (stem + ".PScan." + run_key)
if ps.exists():
    plot_pscan_compression_time(read_pscan(ps))
else:
    print("无 PScan 文件 (NEWRUN 中 Phase_Scan=F)")
sc = SIM_DIR / (stem + ".Scan." + run_key)
if sc.exists():
    scan = read_scan(sc)
    plot_scan_fom(scan, i=0)
    plot_scan_position(scan)
else:
    print("无 Scan 文件 (SCAN namelist 未启用)")
tc = SIM_DIR / (stem + ".tcheck." + run_key)
if tc.exists():
    plot_tcheck_counter(read_tcheck(tc))
else:
    print("无 tcheck 文件 (TcheckS=F)")

In [ ]:
# 菜单 1/2/3/4 补充: 拉莫尔角 / PScan 相位·压缩 / 损失·束载 / 缩放因子 /
# 误差直方图 / 排除 cross-over 发射度·束斑 (均带文件存在守卫)
from pathlib import Path
from astra_tools.io.astra_misc import (read_larmor, read_pscan, read_error,
                                       read_tcheck, read_cr_emit)
from astra_tools.plot.advanced_plots import (
    plot_larmor, plot_phase_scan, plot_pscan_dedz, plot_pscan_compression,
    plot_losses, plot_beam_loading, plot_tcheck_scaling,
    plot_error_hist, plot_cr_emit)
from astra_tools.io import parse_output_file

# R2-2-4: 全部用 cell 2 动态选择的 stem/run_key, 不再硬编码 .001

# 菜单 1 项 13: 拉莫尔角
lm = SIM_DIR / (stem + ".Larmor." + run_key)
if lm.exists():
    plot_larmor(read_larmor(lm))
else:
    print("无 Larmor 文件 (OUTPUT 中 LarmorS=F)")

# 菜单 2 项 1/2/3: PScan 能量 vs 相位 / dE/dz vs 相位 / 压缩因子(z)
ps = SIM_DIR / (stem + ".PScan." + run_key)
if ps.exists():
    _ps = read_pscan(ps)
    plot_phase_scan(_ps)
    plot_pscan_dedz(_ps)
    plot_pscan_compression(_ps)
else:
    print("无 PScan 文件 (NEWRUN 中 Phase_Scan=F)")

# 菜单 2 项 5/6/7: LandF 粒子损失 / 能量沉积 / 束载
lf = SIM_DIR / (stem + ".LandF." + run_key)
if lf.exists():
    _lf = parse_output_file(str(lf))
    plot_losses(_lf)
    plot_beam_loading(_lf)
else:
    print("无 LandF 文件 (OUTPUT 中 LandF_S=F)")

# 菜单 2 项 9: 空间电荷缩放因子
tc = SIM_DIR / (stem + ".tcheck." + run_key)
if tc.exists():
    plot_tcheck_scaling(read_tcheck(tc))
else:
    print("无 tcheck 文件 (TcheckS=F)")

# 菜单 3 项 1-10 右: 误差扫描 FOM 直方图
er = SIM_DIR / (stem + ".Error." + run_key)
if er.exists():
    plot_error_hist(read_error(er), i=0)
else:
    print("无 Error 文件 (误差扫描未启用)")

# 菜单 4 项 12/13: 排除 cross-over 粒子的发射度与束斑
cr = SIM_DIR / (stem + ".Cr_emit." + run_key)
if cr.exists():
    plot_cr_emit(read_cr_emit(str(cr)))
else:
    print("无 Cr_emit 文件 (Cross_start=Cross_end, 未启用)")

In [ ]:
# 菜单 1 项 15 + 菜单 4 项 3/4/5/6: 柱坐标轨迹 / 发射度差 /
# Eq. 4.4 相关贡献 / 缩减纵向发射度 (均带文件存在守卫)
from pathlib import Path
import glob
from astra_tools.io.astra_misc import read_track_file, read_xemit2
from astra_tools.plot.advanced_plots import (
    plot_probe_trajectories, plot_emittance_difference,
    plot_correlated_emittance_contributions, plot_reduced_longitudinal_emittance)
from astra_tools.io import read_distribution

# 菜单 1 项 15: 探针轨迹柱坐标 (r/z + x/y 投影)
tr = SIM_DIR / (stem + ".track." + run_key)
if tr.exists():
    plot_probe_trajectories(tr, mode="cylindrical")
else:
    print("无 track 文件 (TrackS=F, 柱坐标轨迹不可用)")

# 菜单 4 项 3/4/5: 标准-缩减发射度差 + 相关贡献 (Eq. 4.4, 手册 4.13.6)
# 需 Xemit2/Yemit2 文件 (OUTPUT 中 Lsub_cor=T)
x2f = SIM_DIR / (stem + ".Xemit2." + run_key)
y2f = SIM_DIR / (stem + ".Yemit2." + run_key)
if x2f.exists():
    _x2 = read_xemit2(x2f)
    _y2 = read_xemit2(y2f) if y2f.exists() else None
    plot_emittance_difference(emit, _x2, _y2)
    plot_correlated_emittance_contributions(_x2, _y2)
else:
    print("无 Xemit2 文件 (OUTPUT 中 Lsub_cor=F, 缩减发射度不可用)")

# 菜单 4 项 6: 缩减纵向发射度 (当前分布, 减去 2nd/3rd 阶相关)
_dfs = sorted(f for f in glob.glob(str(SIM_DIR / (stem + ".*." + run_key)))
              if len(f.rsplit(".", 2)[1]) == 4 and f.rsplit(".", 2)[1].isdigit())
if _dfs:
    plot_reduced_longitudinal_emittance(read_distribution(_dfs[-1]))
else:
    print("无分布文件, 跳过缩减纵向发射度")
